In [ ]:
!pip install yfinance requests

In [ ]:
!pip install openpyxl

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import csv
import time
import datetime
import requests
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px


sys.path.append(os.path.abspath(".."))
from src.ETF_holdings_enriched import process_universal_etf
from src.look_through_analysis import run_look_through_analysis

from dotenv import load_dotenv
# Load environment variables
load_dotenv()

In [ ]:
%run ../src/portfolio_engine.py

In [ ]:
# input paths/ETF's and output path + file name from .env file -- please also check description.txt for more details
base_dir = Path(os.getenv("data_dir", "."))
output_dir = os.path.join(base_dir, "processed")
etf1_output_csv = Path(output_dir) / "ETF1_holdings_enriched.csv"
etf2_output_csv = Path(output_dir) / "ETF2_holdings_enriched.csv"
etf3_output_csv = Path(output_dir) / "ETF3_holdings_enriched.csv"
etf4_output_csv = Path(output_dir) / "ETF4_holdings_enriched.csv"
input_dir = Path(os.getenv("ETF1_path"))
etf1_input_csv = Path(os.getenv("ETF1"))
etf2_input_csv = Path(os.getenv("ETF2"))
etf3_input_csv = Path(os.getenv("ETF3"))
etf4_input_csv = Path(os.getenv("ETF4"))
etf1_map_file = Path(input_dir) / "name_ticker_manual_map.csv"
consolidated_file = Path(output_dir) / "Consolidated_Portfolio_Positions.csv"

In [ ]:
# Configuration for orchestrator
# Source 1
etf1_ticker = os.getenv("etf1_ticker")
etf1_name = os.getenv("etf1_name")
etf1_config = {
    "index_name": etf1_name,      
    "index_ticker": etf1_ticker,             
    "col_name": "Component Name",                
    "col_isin": None,                
    "col_weight": "Weight %", 
    "col_country": None
}

# Source 2
etf2_ticker = os.getenv("etf2_ticker")
etf2_name = os.getenv("etf2_name")
etf2_config = {
    "index_name": etf2_name,      
    "index_ticker": etf2_ticker,             
    "col_name": "Name",                
    "col_isin": "ISIN",                
    "col_weight": "Weighting", 
    "col_country": "Country"
}

# Source 3
etf3_ticker = os.getenv("etf3_ticker")
etf3_name = os.getenv("etf3_name")
etf3_config = {
    "index_name": etf3_name,      
    "index_ticker": etf3_ticker,             
    "col_name": "Name",                
    "col_isin": "ISIN",                
    "col_weight": "Weighting", 
    "col_country": "Country"
}

# Source 4
etf4_ticker = os.getenv("etf4_ticker")
etf4_name = os.getenv("etf4_name")
Country = os.getenv("etf4_country")
etf4_config = {
    "index_name": etf4_name,      
    "index_ticker": etf4_ticker,             
    "col_name": "Security Name",                
    "col_isin": "ISIN",                
    "col_weight": "Weight (%)", 
    "col_country": Country
}

In [ ]:
# For each etf their holding change half yearly or yearly, when they change files have to be manually downloaded placed in respective folders -- please also check description.txt for more details
# Standardizes raw ETF holdings data by dynamically skipping file headers, & footers, fixing Excel decimal weight formats, and filtering out non-numeric rows.
# Cross-references positions with a manual map and Yahoo Finance APIs to pull tickers, ISINs, sectors, and industries into a structured CSV file.
import datetime

def run_update_orchestrator():
    # 1. Automated 6-Month Check using local file
    date_file = "last_run.txt"
    today = datetime.datetime.now()
    
    if os.path.exists(date_file):
        with open(date_file, "r") as f:
            last_date_str = f.read().strip()
            last_update = datetime.datetime.strptime(last_date_str, "%Y-%m-%d")
    else:
        # If file doesn't exist, set a default past date to trigger the alert
        last_update = today - datetime.timedelta(days=200)
    
    if (today - last_update).days >= 180:
        print("!!! ALERT: It has been 6 months since your last update.")
        print("Please check the ETF providers' websites for new holding files.\n")

    # 2. Define your configurations mapping
    etf_registry = {
        '1': {'config': etf1_config, 'input': etf1_input_csv, 'output': etf1_output_csv, 'map': etf1_map_file},
        '2': {'config': etf2_config, 'input': etf2_input_csv, 'output': etf2_output_csv, 'map': None},
        '3': {'config': etf3_config, 'input': etf3_input_csv, 'output': etf3_output_csv, 'map': None},
        '4': {'config': etf4_config, 'input': etf4_input_csv, 'output': etf4_output_csv, 'map': None}
    }

    # 3. Ask the user for input
    print("Which ETF holdings have you updated? (Enter numbers, e.g., 2,4)")
    print("Type 'q' to quit (no updates today). or Enter 'all' to run all updates.")
    choice = input("Your choice: ").strip().lower()

    if choice == 'q':
        print("No updates selected. Exiting.")
        return

    # 4. Determine which to run
    targets = etf_registry.keys() if choice == 'all' else [c.strip() for c in choice.split(',')]

    # 5. Execute selected targets
    ran_update = False
    for t in targets:
        if t in etf_registry:
            data = etf_registry[t]
            print(f"\n--- Running update for ETF {t} ---")
            process_universal_etf(
                input_path=data['input'],
                output_path=data['output'],
                map_path=data['map'],
                config=data['config']
            )
            ran_update = True
        else:
            print(f"Skipping: ETF {t} is not a valid selection.")
            
    # 6. Update the tracker file if updates were performed
    if ran_update:
        with open(date_file, "w") as f:
            f.write(today.strftime("%Y-%m-%d"))
        print("\nTracker updated. Next 6-month check starts from today.")

run_update_orchestrator() 

In [ ]:
# Looks through the entire portfolio and breaks down all ETF positions into their underlying stock weights.
# Then, combines them with direct holdings and uses config/asset_groups.csv to normalize names via ISIN or name matching.
# Finally, aggregates everything into a master exposure CSV grouped by asset, sector, industry, and country.

etf_mapping = {
    etf1_name: etf1_output_csv,
    etf2_name: etf2_output_csv,
    etf3_name: etf3_output_csv,
    etf4_name: etf4_output_csv
}

final_df = run_look_through_analysis(
    consolidated_file_path=consolidated_file, 
    etf_file_mapping=etf_mapping, 
    output_dir=output_dir
)

In [ ]:
# Ploting the look through analysis
def save_and_plot_treemap(final_df):
    # 1. Storage path for treemap
    chart_dir = Path("../data/charts")
    chart_dir.mkdir(parents=True, exist_ok=True)
    output_path = chart_dir / "portfolio_treemap.html"
    
    # 2. Copy and filter positive values
    plot_df = final_df.copy()
    plot_df = plot_df[plot_df['Total Value (EUR)'].notna() & (plot_df['Total Value (EUR)'] > 0)]
    plot_df['Total Value (EUR)'] = plot_df['Total Value (EUR)'].round(1)
    
    # Clean up textual columns
    columns_to_clean = ['Sector', 'Industry', 'Group Key', 'Security Name']
    for col in columns_to_clean:
        plot_df[col] = plot_df[col].fillna(f"Unknown {col}").astype(str).str.strip()
        plot_df.loc[plot_df[col] == "", col] = f"Unknown {col}"

    if plot_df.empty:
        print("Error: The filtered DataFrame is empty.")
        return

    fig = px.treemap(
        plot_df,
        path=[px.Constant("Total Portfolio"), 'Sector', 'Industry', 'Security Name'], 
        values='Total Value (EUR)',
        color='Sector',
        title="Portfolio Look-Through Exposure — Long Positions Only (i.e Current Value in EUR)",
        hover_data=['Group Key', 'Source']
    )
    
    fig.update_traces(
        textinfo="label+value+percent parent",
        hovertemplate="<b>%{label}</b><br>Value: €%{value:,.1f}<br>Share of Parent: %{percentParent:.2%}<extra></extra>"
    )
    
    fig.update_layout(
        margin=dict(t=50, l=10, r=10, b=10),
        height=700
    )
    
    # 5. Save and render
    fig.write_html(str(output_path), include_plotlyjs='cdn')
    print(f"Success! Properly formatted chart saved to: {output_path}")
    
    return fig.show(renderer="notebook_connected")

# Run it
save_and_plot_treemap(final_df)